In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from src.pipeline import *
from src.models.neural_models import *

# EDA and Visualizations

In [ ]:
data=pd.read_csv('./data/processed/final_dataset_with_indicators.csv')
data.head()

In [ ]:
plt.plot(data['Date'], data['Ucome_fob_ARA']);

In [ ]:
plt.boxplot(data['Ucome_fob_ARA'])

# Model Training Example

In [ ]:
# Split first, then fit scaler/selector on train only
train_selected, test_selected, selected_cols = split_scale_feature_select(scaler_type='MinMax')
scaled_data = pd.concat([train_selected, test_selected], ignore_index=True)
scaled_data.head()

In [ ]:
target_col = scaled_data['Ucome_fob_ARA']

In [ ]:
# Use leakage-safe outputs from split_scale_feature_select (Cell 7)
# Do NOT run feature selection again on combined train+test.

# Update the number of features after train-only feature selection
n_features = train_selected.shape[1]  # includes target column

# Use the pre-split selected sets directly
train_data = train_selected.copy()
test_data = test_selected.copy()

# Prepare the data for the neural network
window_size = 30
lookahead_value = 10
batch_size = 32

In [ ]:
windowed_train = window_creator(train_data, window_size, lookahead_value, batch_size)

windowed_test = window_creator(test_data, window_size, lookahead_value, batch_size)

In [ ]:
# Optimize hyperparameters
ltsm_params = optimize_hyperparameters(
    train_df=train_data, 
    model_type='lstm',
    lookahead_value=lookahead_value, 
    batch_size=batch_size,
    window_size=window_size,
    n_features=n_features
)

gru_params = optimize_hyperparameters(
    train_df=train_data,
    model_type='gru',
    lookahead_value=lookahead_value,
    batch_size=batch_size,
    window_size=window_size,
    n_features=n_features
)

hybrid_params = optimize_hyperparameters(
    train_df=train_data, 
    model_type='hybrid',
    lookahead_value=lookahead_value, 
    batch_size=batch_size,
    window_size=window_size,
    n_features=n_features
)


In [ ]:
#models
ltsm_model = LSTMModel(
    lstm_units=int(hybrid_params['lstm_units']),
    dropout=float(hybrid_params['dropout']),
    learning_rate=float(hybrid_params['learning_rate']),
    window_size=window_size,
    n_features=n_features
)
gru_model = GRUModel(
    gru_units=int(hybrid_params['gru_units']),
    dropout=float(hybrid_params['dropout']),
    learning_rate=float(hybrid_params['learning_rate']),
    window_size=window_size,
    n_features=n_features
)
hybrid_model = HybridGRU_LSTM(
    gru_units=int(hybrid_params['gru_units']),
    lstm_units=int(hybrid_params['lstm_units']),
    dropout=float(hybrid_params['dropout']),
    learning_rate=float(hybrid_params['learning_rate']),
    window_size=window_size,
    n_features=n_features
)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.feature_engine import technical_indicator_creation

print("Training model on training set...")
history = hybrid_model.train(windowed_train)
print("Training complete!\n")

print("Evaluating model on test set...")
test_metrics = hybrid_model.evaluate(windowed_test)
print(f"Test Loss (MSE, scaled): {test_metrics[0]:.6f}")
print(f"Test MAE (scaled): {test_metrics[1]:.6f}\n")

print("Generating predictions on test set...")
predictions = hybrid_model.predict(windowed_test).flatten()

# Rebuild y_true from windowed_test to avoid shape/alignment mistakes
y_true_scaled = []
for _, y_batch in windowed_test:
    y_true_scaled.extend(y_batch.numpy().flatten())
y_true_scaled = pd.Series(y_true_scaled).to_numpy()

# Scaled-space metrics
scaled_mae = mean_absolute_error(y_true_scaled, predictions)
scaled_rmse = mean_squared_error(y_true_scaled, predictions) ** 0.5
scaled_r2 = r2_score(y_true_scaled, predictions)

print(f"Predictions shape: {predictions.shape}")
print(f"First 10 predictions:\n{predictions[:10]}")
print(f"Prediction range: [{predictions.min():.4f}, {predictions.max():.4f}]")
print(f"Scaled MAE: {scaled_mae:.6f}")
print(f"Scaled RMSE: {scaled_rmse:.6f}")
print(f"Scaled R2: {scaled_r2:.6f}")

# Convert to original price units for interpretability
raw_target = technical_indicator_creation()["Ucome_fob_ARA"].reset_index(drop=True)
y_min = raw_target.min()
y_max = raw_target.max()
scale = (y_max - y_min)

y_true_orig = y_true_scaled * scale + y_min
y_pred_orig = predictions * scale + y_min

orig_mae = mean_absolute_error(y_true_orig, y_pred_orig)
orig_rmse = mean_squared_error(y_true_orig, y_pred_orig) ** 0.5
orig_r2 = r2_score(y_true_orig, y_pred_orig)

print(f"Original-unit MAE: {orig_mae:.6f}")
print(f"Original-unit RMSE: {orig_rmse:.6f}")
print(f"Original-unit R2: {orig_r2:.6f}")
print("\nModel training and evaluation complete!")

# Linear Models Evaluation

In [ ]:
from src.models.linear_models import run_linear_models_experiment

In [ ]:
raw_data = technical_indicator_creation().copy()

# Use the exact RF-selected feature subset
feature_cols = [c for c in selected_cols if c in raw_data.columns]

linear_results_df, linear_artifacts = run_linear_models_experiment(
    data=raw_data,
    target_col='Ucome_fob_ARA',
    feature_cols=feature_cols,
    horizon=1,
    target_mode='level',
    start_date='2023-01-01',
    train_size=0.70,
    val_size=0.15,
)

print(f"Using {len(feature_cols)} selected features in linear models.")
linear_results_df

Calculating Indicators for HVO_class_II_fob_ARA
Calculating Indicators for Ucome_fob_ARA
Calculating Indicators for UCO_exw_ARA
Calculating Indicators for EU_BRENT_CRUDE
Calculating Indicators for LSMGO_Rotterdam

All 26 technical indicators calculated successfully!

Indicators added (130) total columns:

DataFrame shape: (1824, 139)


/mnt/c/Code/ML-Final-Project/src/feature_engine.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_price_df[f'{column}_RoC_12'] = stock['close_12_roc']  # Rate of Change: 12-period
/mnt/c/Code/ML-Final-Project/src/feature_engine.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_price_df[f'{column}_Coppock_Curve'] = stock['coppock']  # Coppock Curve
/mnt/c/Code/ML-Final-Project/src/feature_engine.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti

Prepared model dataset: X=(1822, 457), y=(1822,)
Target mode: level, horizon: 1
Feature count: 457
Train: (1275, 457)
Validation: (273, 457)
Test: (274, 457)
Train dates: 2023-01-01 00:00:00 to 2026-06-28 00:00:00
Validation dates: 2026-06-29 00:00:00 to 2027-03-28 00:00:00
Test dates: 2027-03-29 00:00:00 to 2027-12-27 00:00:00


/mnt/c/Code/ML-Final-Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.194e+04, tolerance: 1.566e+04
  model = cd_fast.enet_coordinate_descent(


Using 56 selected features in linear models.


/mnt/c/Code/ML-Final-Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.110e+05, tolerance: 1.566e+04
  model = cd_fast.enet_coordinate_descent(


,model,MAE,RMSE,R2,Directional Accuracy,split
0,Naive Baseline,8.964176,13.369932,0.871166,0.000000,validation
1,Linear Regression,187.201476,225.952339,-35.796431,0.501832,validation
2,Ridge,44.232401,59.343199,-1.538130,0.505495,validation
3,Lasso,44.218613,60.565270,-1.643744,0.538462,validation
4,Elastic Net,33.589933,49.649041,-0.776617,0.472527,validation
5,Naive Baseline,8.198175,13.420881,0.911658,0.003650,test
6,Linear Regression,155.170471,239.045683,-27.026437,0.485401,test
7,Ridge,56.931456,68.646685,-1.311241,0.485401,test
8,Lasso,67.028504,76.002490,-1.833099,0.496350,test
9,Elastic Net,33.649819,48.798428,-0.167933,0.540146,test
